# SDK prunding


### Local SDK Path


In [ ]:
SDK_PATH = '/root/autodl-tmp/revitdocs/Samples'

## Get ReadMe Doc

In [20]:
import os
import json
from striprtf.striprtf import rtf_to_text
from dotenv import load_dotenv
from openai import OpenAI
import google.generativeai as genai

def read_readme_doc(path: str) -> dict:
    with open(path, 'r', encoding='utf-8', errors='ignore') as f:
        content = f.read()

    # make rtf doc -> text doc 
    plain_text_content = rtf_to_text(content)

    # The f-string prompt with escaped curly braces in the JSON example
    prompt_sdk_select = f"""
        # ROLE
        You are an AI assistant specializing in codebase analysis, an expert at extracting structured data from technical documentation.

        # GOAL
        Your goal is to accurately parse the provided ReadMe file to extract key identifiers for code. This output will be used programmatically by an automated code retrieval and analysis system, so the accuracy and format of your response are critical.

        # INSTRUCTIONS
        1.  Carefully analyze the text provided within the `<ReadMeContent>` tags.
        2.  Extract the following three categories of information:
            - `target_files`: A list of all project source filenames (e.g., `.cs` files) explicitly mentioned in the text.
            - `key_classes_and_methods`: A list of the names of custom classes or methods created *within* the project that are identified as being responsible for core functionality.
            - `mentioned_apis`: A list of key API classes from external frameworks or libraries (e.g., `Autodesk.Revit.DB.View`) that are explicitly listed in the text.
        3.  Format your output as a single, strict JSON object.
        4.  If no information is found for a specific field, its value must be an empty list (`[]`). Do not omit the key from the JSON object.
        5.  Your final response **MUST** contain *only* the raw JSON object, without any explanatory text, markdown code blocks, or other conversational filler.

        # EXAMPLE
        <ExampleReadMe>
        Summary: This tool is in the file `Processor.cs`. The core logic is handled by the `DataParser` class, which uses the `Autodesk.Revit.DB.Transaction` API.
        </ExampleReadMe>
        <ExampleJSONOutput>
        {{
        "target_files": ["Processor.cs"],
        "key_classes_and_methods": ["DataParser"],
        "mentioned_apis": ["Autodesk.Revit.DB.Transaction"]
        }}
        </ExampleJSONOutput>

     
    """

    # initialize openai
    load_dotenv(dotenv_path='/root/autodl-tmp/python_revit_train/gemini_api.env')
    # Corrected environment variable name for consistency
    deepseek_api_key = os.getenv("DEEPSEEK_API_KEY")
    if not deepseek_api_key:
        raise ValueError("Error: DEEPSEEK_API_KEY environment variable not set.")
        
    #genai.configure(api_key=gemini_api_key)

    #model = genai.GenerativeModel('gemini-1.5-flash-latest')

    #response = model.generate_content(prompt_sdk_select)

    # Create Response
    client = OpenAI(api_key=deepseek_api_key , base_url="https://api.deepseek.com")

    response = client.chat.completions.create(
    model="deepseek-chat",
        messages=[
            {"role": "system", "content": prompt_sdk_select},  
            
            {"role": "user", "content": f"{plain_text_content}"}
        ],
        stream = False
    )

    # response = gemini_model.generate_content(query_llm)

    # print(f"query: {query_llm}")
    print("Response from DeepSeek:")
    print(response.choices[0].message.content)



    cleaned_json_string = response.choices[0].message.content.strip().replace("```json", "").replace("```", "").strip()

    return json.loads(cleaned_json_string)


if __name__ == "__main__":
    # Ensure the striprtf library is installed: pip install striprtf
    target_content = read_readme_doc('/root/autodl-tmp/revitdocs/Samples/AllViews/CS/ReadMe_AllViews.rtf')
    print(json.dumps(target_content, indent=2, ensure_ascii=False))

Response from DeepSeek:
{
  "target_files": ["AllViews.cs", "AllViewsForm.cs"],
  "key_classes_and_methods": ["Command", "ViewsMgr", "AllViewsForm"],
  "mentioned_apis": ["Autodesk.Revit.DB.View", "Autodesk.Revit.DB.ViewSet", "Autodesk.Revit.Creation.Document.NewViewSheet"]
}
{
  "target_files": [
    "AllViews.cs",
    "AllViewsForm.cs"
  ],
  "key_classes_and_methods": [
    "Command",
    "ViewsMgr",
    "AllViewsForm"
  ],
  "mentioned_apis": [
    "Autodesk.Revit.DB.View",
    "Autodesk.Revit.DB.ViewSet",
    "Autodesk.Revit.Creation.Document.NewViewSheet"
  ]
}


## Get This Project Most Important Code Setences

In [85]:
from tree_sitter import Language , Parser , Query , QueryCursor
import tree_sitter_c_sharp

CSHARP_LANGUAGE =  Language(tree_sitter_c_sharp.language())

csharp_code_for_query = """
public class Calculator
{
    public int Add(int x, int y) => x + y;
    private static string GetWelcomeMessage() => "Welcome!";
}
"""



cpp_parser = Parser(CSHARP_LANGUAGE)


tree = cpp_parser.parse(bytes(csharp_code_for_query, "utf8"))
root_node = tree.root_node

# 定义一个查询字符串
# - 查找所有 method_declaration 节点
# - 在该节点下，捕获返回类型 (predefined_type 或 identifier) 并命名为 @return.type
# - 捕获方法名 (identifier) 并命名为 @method.name
# https://tree-sitter.github.io/tree-sitter/7-playground.html
query_string = """
(method_declaration
  returns: (_) @return.type
  name: (identifier) @method.name
)

(parameter
  type: (_) @param.type
  name: (identifier) @param.name
)
"""

# 创建查询对象
query = Query(CSHARP_LANGUAGE, query_string)
cursor = QueryCursor(query)
# 对语法树执行查询
captures = cursor.captures(root_node)
print(captures)
# captures 是一个元组列表，每个元组包含 (node, capture_name)
print("--- 使用新的 for 循环逻辑进行处理 ---")
methods_data = {}
current_method_details = None
last_param_type = None

# 遍历扁平的捕获列表
for node, capture_name in captures:
    node_text = node.text.decode('utf8')

    # 当我们找到一个方法名时，意味着一个新的方法开始了
    if capture_name == 'method.name':
        # 创建一个新的字典来存储这个方法的信息
        current_method_details = {
            "name": node_text,
            "return_type": "未知", # 先设置一个默认值
            "params": []
        }
        # 使用方法名作为主字典的键
        methods_data[node_text] = current_method_details

    # 返回类型通常紧跟在方法名之前，但为简化，我们在这里处理
    # 这个逻辑依赖于返回类型总是在方法名之前被捕获
    elif capture_name == 'return.type':
        # 如果我们已经创建了一个方法详情字典，就更新它的返回类型
        if current_method_details:
            current_method_details['return_type'] = node_text

    # 处理参数类型
    elif capture_name == 'param.type':
        # 暂存参数类型，等待参数名出现
        last_param_type = node_text
    
    # 处理参数名
    elif capture_name == 'param.name':
        # 确保我们正在一个方法的上下文中，并且刚刚看到了一个参数类型
        if current_method_details and last_param_type:
            current_method_details['params'].append(f"{last_param_type} {node_text}")
            last_param_type = None # 使用后重置

# 打印最终结果
for name, details in methods_data.items():
    return_type = details.get('return_type', '未知类型')
    params_list = details.get('params', [])
    params_str = ", ".join(params_list) if params_list else "无"

    print(f"方法名: {name}, 返回类型: {return_type}, 参数: {params_str}")

print("--- 查询结束 ---")

{'return.type': [<Node type=predefined_type, start_point=(3, 11), end_point=(3, 14)>, <Node type=predefined_type, start_point=(4, 19), end_point=(4, 25)>], 'method.name': [<Node type=identifier, start_point=(3, 15), end_point=(3, 18)>, <Node type=identifier, start_point=(4, 26), end_point=(4, 43)>], 'param.type': [<Node type=predefined_type, start_point=(3, 19), end_point=(3, 22)>, <Node type=predefined_type, start_point=(3, 26), end_point=(3, 29)>], 'param.name': [<Node type=identifier, start_point=(3, 23), end_point=(3, 24)>, <Node type=identifier, start_point=(3, 30), end_point=(3, 31)>]}
--- 使用新的 for 循环逻辑进行处理 ---


ValueError: too many values to unpack (expected 2)

In [ ]:
import json
from tree_sitter import Language , Parser


